# Explore: Polars documentation health

Worked example of visualizing `extract_docs()` output with seaborn.

Clone Polars into this `examples/` folder first:

```
cd examples
git clone https://github.com/pola-rs/polars
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.append(str(Path("..").resolve()))
from doc_health import extract_docs
from polars_config import CONFIG

# Palette roles - see the dataviz skill's reference palette for the source values.
SURFACE = "#fcfcfb"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRIDLINE = "#e1e0d9"
SEQUENTIAL_BLUE = "#2a78d6"  # single-hue fill for magnitude/distribution charts

# Fixed categorical order (never cycled/reassigned) for the four Diataxis types.
DIATAXIS_COLORS = {
    "tutorial": "#2a78d6",     # slot 1 blue
    "how-to": "#eb6834",       # slot 2 orange
    "reference": "#1baf7a",    # slot 3 aqua
    "explanation": "#eda100",  # slot 4 yellow
}

sns.set_theme(style="white", rc={
    "axes.facecolor": SURFACE,
    "figure.facecolor": SURFACE,
    "axes.edgecolor": GRIDLINE,
    "axes.labelcolor": INK_SECONDARY,
    "text.color": INK_PRIMARY,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "grid.color": GRIDLINE,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
repo = Path("polars")
df = extract_docs(repo, CONFIG)

# Left join, no assert here: this is exploration, not a validation gate.
# polars_diataxis_types.csv covers all pages under user-guide/, but only the
# 14 under expressions/ were read and labeled by hand - the rest were seeded
# by suggest_diataxis.py and haven't been reviewed (see its docstring).
diataxis_df = pd.read_csv("polars_diataxis_types.csv")
df = df.merge(diataxis_df, on="path", how="left")
print(f"{df['diataxis_type'].notna().sum()} of {len(df)} pages have a diataxis label")
df.describe(include="all")

In [ ]:
# Distributions first - staleness, churn, and length are typically heavily
# right-skewed for a real docs set, not normal, so look at shape before summary stats.
metrics = [
    ("days_since_update", "Days since update (staleness)"),
    ("word_count", "Word count"),
    ("code_block_density", "Code examples per 1000 words"),
    ("heading_max_depth", "Heading max depth"),
    ("commit_count", "Commit count (churn)"),
    ("flesch_reading_ease", "Flesch reading ease"),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (col, title) in zip(axes.flat, metrics):
    sns.histplot(df[col], color=SEQUENTIAL_BLUE, ax=ax)
    ax.set_title(title, color=INK_PRIMARY, fontsize=11)
    ax.set_xlabel("")
    ax.grid(axis="y", linewidth=0.5)
fig.tight_layout()

In [ ]:
# Diataxis breakdown - only meaningful for the labeled subset right now, and
# that subset is small and skewed (see the print output two cells up), so
# treat this as a shape check, not a finding, until the CSV is fully reviewed.
labeled = df.dropna(subset=["diataxis_type"])
order = [t for t in DIATAXIS_COLORS if t in labeled["diataxis_type"].unique()]

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(
    data=labeled, x="diataxis_type", hue="diataxis_type", order=order,
    palette=[DIATAXIS_COLORS[t] for t in order], legend=False, ax=ax,
)
ax.set_title("Pages by Diataxis type (labeled subset)", color=INK_PRIMARY, fontsize=11)
ax.set_xlabel("")
ax.grid(axis="y", linewidth=0.5)

In [ ]:
# Staleness by Diataxis type, for the labeled subset - a first look at whether
# any type skews stale. Small-n per group right now (see caveat above).
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=labeled, x="diataxis_type", y="days_since_update", hue="diataxis_type", order=order,
    palette=[DIATAXIS_COLORS[t] for t in order], legend=False, ax=ax,
)
sns.stripplot(
    data=labeled, x="diataxis_type", y="days_since_update", order=order,
    color=INK_SECONDARY, alpha=0.6, ax=ax,
)
ax.set_title("Staleness by Diataxis type", color=INK_PRIMARY, fontsize=11)
ax.set_xlabel("")
ax.grid(axis="y", linewidth=0.5)